# Keystroke Feature Prep

This notebook loads Android Room keystroke events from SQLite, converts timestamps, groups by session, and returns a clean pandas DataFrame ready for feature engineering.

## 2. Connect to the SQLite Database

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

## 8. 30-Day Typing Baseline Dashboard

## 9. Isolation Forest Anomaly Detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def build_baseline_dashboard(daily_features: pd.DataFrame, output_path: str = "baseline_report.png") -> plt.Figure:
    """Build and save a 30-day typing baseline dashboard."""
    sns.set_theme(style="whitegrid", context="talk")

    plot_frame = daily_features.copy()
    plot_frame.index = pd.to_datetime(plot_frame.index)
    plot_frame = plot_frame.sort_index().tail(30)

    numeric_frame = plot_frame.select_dtypes(include=["number", "float", "int"]).copy()
    correlation_matrix = numeric_frame.corr(numeric_only=True)

    fig, axes = plt.subplots(2, 3, figsize=(20, 11), constrained_layout=True)
    fig.suptitle("30-Day Typing Baseline Dashboard", fontsize=20, fontweight="bold")

    axes[0, 0].plot(plot_frame.index, plot_frame["avg_iki"], color="#1f77b4", linewidth=2, marker="o")
    axes[0, 0].set_title("Avg IKI Over Time")
    axes[0, 0].set_ylabel("ms")
    axes[0, 0].tick_params(axis="x", rotation=45)

    backspace_threshold = plot_frame["backspace_rate"].mean() + plot_frame["backspace_rate"].std(ddof=0)
    axes[0, 1].plot(plot_frame.index, plot_frame["backspace_rate"], color="#d62728", linewidth=2, marker="o")
    axes[0, 1].axhline(backspace_threshold, color="red", linestyle="--", linewidth=2, label=f"mean+1std = {backspace_threshold:.3f}")
    axes[0, 1].set_title("Backspace Rate Over Time")
    axes[0, 1].set_ylabel("rate")
    axes[0, 1].legend(loc="best")
    axes[0, 1].tick_params(axis="x", rotation=45)

    axes[0, 2].plot(plot_frame.index, plot_frame["wpm_estimate"], color="#2ca02c", linewidth=2, marker="o")
    axes[0, 2].set_title("WPM Estimate Over Time")
    axes[0, 2].set_ylabel("wpm")
    axes[0, 2].tick_params(axis="x", rotation=45)

    axes[1, 0].bar(plot_frame.index, plot_frame["late_night_ratio"], color="#9467bd")
    axes[1, 0].set_title("Late Night Ratio")
    axes[1, 0].set_ylabel("fraction")
    axes[1, 0].tick_params(axis="x", rotation=45)

    sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, annot=True, fmt=".2f", ax=axes[1, 1])
    axes[1, 1].set_title("Feature Correlation Heatmap")

    avg_iki_values = plot_frame["avg_iki"].dropna().astype(float)
    axes[1, 2].hist(avg_iki_values, bins=12, density=True, alpha=0.75, color="#1f77b4", edgecolor="white")
    if len(avg_iki_values) > 1:
        mean_iki = avg_iki_values.mean()
        std_iki = avg_iki_values.std(ddof=0)
        if std_iki > 0:
            x = np.linspace(avg_iki_values.min(), avg_iki_values.max(), 200)
            normal_curve = (1 / (std_iki * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean_iki) / std_iki) ** 2)
            axes[1, 2].plot(x, normal_curve, color="black", linewidth=2)
    axes[1, 2].set_title("Avg IKI Distribution")
    axes[1, 2].set_xlabel("ms")
    axes[1, 2].set_ylabel("density")

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return fig


# Run after building daily_features:
# daily_features = build_daily_features(feature_ready_df)
# build_baseline_dashboard(daily_features)

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


def train_baseline_anomaly_model(
    daily_features: pd.DataFrame,
    model_path: str = "wellness_model.pkl",
    scaler_path: str = "scaler.pkl",
    contamination: float = 0.1,
) -> pd.DataFrame:
    """Train an Isolation Forest on daily typing features and annotate anomalies.

    The feature contributions printed for anomaly days are heuristic: they use the
    largest absolute standardized deviations from the baseline distribution.
    """
    feature_columns = [
        "avg_iki",
        "std_iki",
        "backspace_rate",
        "avg_hold",
        "wpm_estimate",
        "late_night_ratio",
        "session_count",
        "avg_session_duration",
    ]

    frame = daily_features.copy().sort_index()
    frame.index = pd.to_datetime(frame.index)

    missing_columns = [column for column in feature_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"daily_features is missing required columns: {missing_columns}")

    feature_frame = frame[feature_columns].apply(pd.to_numeric, errors="coerce")
    feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True))

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(feature_frame)

    model = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=200,
    )
    model.fit(scaled_features)

    raw_scores = model.decision_function(scaled_features)
    anomaly_score = (-raw_scores).astype(float)
    is_anomaly = model.predict(scaled_features) == -1

    result = frame.copy()
    result["anomaly_score"] = anomaly_score
    result["is_anomaly"] = is_anomaly.astype(bool)

    joblib.dump(model, model_path)
    joblib.dump(scaler, scaler_path)

    score_frame = result.reset_index().rename(columns={result.index.name or "index": "date"})
    date_column = score_frame.columns[0] if "date" not in score_frame.columns else "date"
    if date_column != "date":
        score_frame = score_frame.rename(columns={date_column: "date"})
    score_frame["date"] = pd.to_datetime(score_frame["date"])

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(score_frame["date"], score_frame["anomaly_score"], color="#1f77b4", linewidth=2, marker="o", label="Anomaly score")
    anomaly_days = score_frame[score_frame["is_anomaly"]]
    ax.scatter(anomaly_days["date"], anomaly_days["anomaly_score"], color="red", s=70, label="Anomaly day", zorder=3)
    ax.set_title("Daily Anomaly Scores")
    ax.set_xlabel("Date")
    ax.set_ylabel("Anomaly score (higher = more anomalous)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    scaled_df = pd.DataFrame(scaled_features, index=frame.index, columns=feature_columns)
    for day in score_frame.loc[score_frame["is_anomaly"], "date"]:
        day_index = pd.Timestamp(day)
        if day_index not in scaled_df.index:
            continue
        contributions = scaled_df.loc[day_index].abs().sort_values(ascending=False)
        top_features = contributions.head(3)
        contribution_text = ", ".join(f"{feature} (|z|={value:.2f})" for feature, value in top_features.items())
        print(f"{day_index.date()}: top contributing features -> {contribution_text}")

    return result


# Example usage:
# anomaly_results = train_baseline_anomaly_model(daily_features)
# anomaly_results.head()